<a href="https://colab.research.google.com/github/Bovi-analytics/Ansci4040FinalClassProject/blob/main/PipelineSteps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Calving-pen pipeline: all steps, one or more videos

One notebook for the whole pipeline, **one cell per step**. Put the videos in the list in cell 2; **each step cell then runs that step for every video in the list**, one after the other, and shows its results. Run the cells in order and check each result before the next.

### Before you start (once per person)
- Access to the Shared Drive **BeastsAndBytes** > `ANSC4040GroupProject`.
- Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.
- A Hugging Face token for DINOv3 (step 05) in Colab Secrets as `HF_TOKEN`, with notebook access on: steps and links in [`CalvingPen/README.md`](https://github.com/Bovi-analytics/Ansci4040FinalClassProject/blob/main/CalvingPen/README.md#hugging-face-token-for-dinov3-step-05).
- Tell the group which videos you run: **one person per video**.

### What to set in cell 2
| Setting | What it is | Where to get it |
|---|---|---|
| `Videos` | one line per video: `VideoFile` (raw video's file name), `VideoStem` (output folder, short: date + pen), `ClipStart` / `ClipEnd` (where the cow is, HH:MM:SS) | the table cell 2 prints; the group log; `Timeline.jpg` from cell 3 |
| `Only` | `None` = every video in the list; or a list of `VideoStem`s to run only those | e.g. to run one video per Colab session |
| `Redo` | `False` = skip steps already finished for a video (continue after a disconnect); `True` = redo every step (overwrite) | keep `False` |
| `Test` | `True` = only 4 minutes ("a" folders), `False` = the whole clip ("b" folders) | `False` once the settings are known |

### Cells
| Cell | Step | Function | Model | T4 time per hour of clip |
|---|---|---|---|---|
| 3 | 01 cut the clip | `PrepareClips` | ffmpeg | a few min |
| 4 | 02 detect | `DetectCows` | **YOLO26n** | ~8 min |
| 5 | 03 track | `TrackCows` | **SAM 2.1 Small**, video mode | **~75 min**, resumable |
| 6 | 04 crop + mask | `CropCows` | **SAM 2.1 Small**, image mode | ~7 min |
| 7 | 05 fingerprint | `EmbedCrops` | **DINOv3 ViT-S/16** | ~1 min |
| 8 | 06 pool (all video folders) | `BuildPool` | none | seconds |

**If Colab stops:** run cells 1–2 again, then the cell that stopped. Finished steps are skipped and step 03 continues from its last finished 2-minute chunk. Every step still **overwrites** its folder when its input or settings changed (or with `Redo = True`). For many videos without looking at each step, `RunPipeline.ipynb` does the same in one cell. The folder is shared: one person runs a given video at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install the model libraries.
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/Bovi-analytics/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline import PrepareClips, SaveTimelineSheet
from CalvingPipeline.Detect import DetectCows
from CalvingPipeline.Track import TrackCows
from CalvingPipeline.Crop import CropCows
from CalvingPipeline.Embed import EmbedCrops
from CalvingPipeline.Pool import BuildPool
from CalvingPipeline.Pipeline import StepDone, StepFolders, VideoStatus
from CalvingPipeline.Paths import FindVideo, ListVideos, OutputRoot
import pandas as pd
from IPython.display import Image, Video, display
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

try:
    HfToken = userdata.get("HF_TOKEN")   # from Colab Secrets; never paste the token here
except Exception:
    HfToken = None
print("HF_TOKEN:", "found" if HfToken else "NOT SET - step 05 (DINOv3) will stop; add it in Colab Secrets")

In [ ]:
# 2. Settings: the only cell to edit.
display(ListVideos())    # the calving videos on the Shared Drive (HasOutputs = someone already ran this video)

# --- Which videos: one line each (file name, output folder, where the cow is) ---
Videos = [
    dict(VideoFile="2026-11-11 00-00-00 Calving Pen 3.avi", VideoStem="2026-11-11 Calving Pen 3",
         ClipStart="00:00:00", ClipEnd="03:04:49"),
    dict(VideoFile="2026-11-11 00-00-00 Calving Pen 2.avi", VideoStem="2026-11-11 Calving Pen 2",
         ClipStart="00:00:00", ClipEnd="04:16:46"),
    # dict(VideoFile="...avi", VideoStem="YYYY-MM-DD Calving Pen N", ClipStart="HH:MM:SS", ClipEnd="HH:MM:SS"),
]
Only = None              # None = all videos above; or e.g. ["2026-11-11 Calving Pen 3"] to run just that one
Redo = False             # False = skip steps already finished for a video; True = redo (overwrite) everything

# --- Test or full ---
Test = False             # True = only 4 minutes from TestStart ("a" folders); False = the whole clip ("b" folders)
TestStart = 1080         # seconds into the clip, only for Test = True

# --- Step settings: keep the defaults unless you are testing a change ---
TargetFps = 2            # 02: frames per second looked at (the camera updates ~2 times a second)
ImageSize = 1024         # 02: YOLO picture size (640 misses the lying cow far more often)
Confidence = 0.25        # 02: lowest YOLO confidence kept
SeedConfidence = 0.4     # 03: YOLO confidence needed to start following a new animal (the cow is carried over)
ChunkSeconds = 120       # 03: SAM's memory is restarted every 2 min (from where it left the cow)
Half = False             # 03: True = half precision on the GPU, faster; check it with Test = True first
Preview = True           # 02 (test only) and 03: preview videos
MinFrames, MinConfidence = 10, 0.4   # 04: drop tracks shorter than 5 s or less sure than 0.4
CropsPerTrack = 8 if Test else 24    # 04: at least this many crops per track
CropEverySeconds = 15    # 04: and at least one every 15 s
GroupSeconds = 60 if Test else 300   # 05: heatmap pieces of this many seconds
ExcludeSeconds = 60 if Test else 300 # 06: self-check may match the same track this far away

# --- Folders (nothing to edit) ---
Run = [V for V in Videos if Only is None or V["VideoStem"] in Only]
Folder = {Step: Name for Step, Name in StepFolders(Test).items()}
Start, End = (TestStart, TestStart + 240) if Test else (0, None)
def Dirs(V):
    VideoDir = OutputRoot / V["VideoStem"]
    return VideoDir, VideoDir / "ReferenceClip.mp4"
def Skip(Step, V, **Settings):
    if not Redo and StepDone(Step, Dirs(V)[0], Test, **Settings):
        print(f"{V['VideoStem']}: step 0{Step} already done, skipped (Redo = True to redo it)")
        return True
    print(f"\n=== {V['VideoStem']}: step 0{Step} ===")
    return False
for V in Run:
    print(V["VideoStem"], "| clip", "cut" if Dirs(V)[1].exists() else "not cut yet", "| outputs:", Dirs(V)[0])

In [ ]:
# 3. Step 01 · Cut the clip (ffmpeg), for each video. Saves Timeline.jpg (one picture every 15 min) first.
for V in Run:
    VideoDir, ClipPath = Dirs(V)
    if Skip(1, V):
        continue
    VideoPath = FindVideo(V["VideoFile"])
    SaveTimelineSheet(VideoPath, VideoDir / "Timeline.jpg", EveryMinutes=15)
    display(Image(str(VideoDir / "Timeline.jpg"), width=1200))
    Summary = PrepareClips(VideoPath, VideoDir, V.get("ClipStart", 0), V.get("ClipEnd"))
    Clip = Summary["ReferenceClip"]
    print(f"Clip {Summary['ClipStart']}-{Summary['ClipEnd']}: {Clip['Duration']}, {Clip['Fps']} fps, "
          f"{Clip['Width']}x{Clip['Height']}  (check: ~30 fps, length as expected)")

In [ ]:
# 4. Step 02 · Detect cows (YOLO26n), for each video.
for V in Run:
    VideoDir, ClipPath = Dirs(V)
    if Skip(2, V, TargetFps=TargetFps, ImageSize=ImageSize, Confidence=Confidence, StartSeconds=Start):
        continue
    DetectCows(ClipPath, VideoDir / Folder[2], Start, End, TargetFps=TargetFps, Confidence=Confidence,
               ImageSize=ImageSize, Preview=Preview and Test)
    Summary = pd.read_csv(VideoDir / Folder[2] / "DetectionSummary.csv")
    print(f"cow seen in {Summary.FramesWithCow.sum() / Summary.SampledFrames.sum():.0%} of frames; "
          f"minutes with CowShare < 0.5: {list(Summary.ClipTime[Summary.CowShare < 0.5])}")

In [ ]:
# 5. Step 03 · Track the cows (SAM 2.1, video mode), for each video. Rerun to continue a stopped run.
for V in Run:
    VideoDir, ClipPath = Dirs(V)
    print(f"\n=== {V['VideoStem']}: step 03 ===")
    TrackCows(ClipPath, VideoDir / Folder[2], VideoDir / Folder[3], ChunkSeconds=ChunkSeconds,
              SeedConfidence=SeedConfidence, CompareTracksPath=VideoDir / "BaselineByteTrack" / "Tracks.csv",
              Half=Half, Preview=Preview, RedoFinished=Redo)
    display(pd.read_csv(VideoDir / Folder[3] / "TrackSummary.csv"))
    print("Preview (open in Drive):", VideoDir / Folder[3] / "PreviewTracks.mp4")

In [ ]:
# 6. Step 04 · Cut out the cows (SAM 2.1, image mode): masked crops, for each video.
for V in Run:
    VideoDir, ClipPath = Dirs(V)
    if Skip(4, V):
        continue
    CropCows(ClipPath, VideoDir / Folder[3] / "Tracks.csv", VideoDir / Folder[4], MinFrames=MinFrames,
             MinConfidence=MinConfidence, CropsPerTrack=CropsPerTrack, CropEverySeconds=CropEverySeconds)
    display(pd.read_csv(VideoDir / Folder[4] / "KeptTracks.csv"))
    display(Image(str(VideoDir / Folder[4] / "Sheets" / "Overview.jpg")))

In [ ]:
# 7. Step 05 · Fingerprint each crop (DINOv3), for each video. No DINOv2 fallback: stops if the token fails.
for V in Run:
    VideoDir, ClipPath = Dirs(V)
    if Skip(5, V):
        continue
    EmbedCrops(VideoDir / Folder[4], VideoDir / Folder[5], V["VideoStem"], Token=HfToken, FallbackModelId=None,
               GroupSeconds=GroupSeconds)
    display(Image(str(VideoDir / Folder[5] / "TrackSimilarity.png")))

In [ ]:
# 8. Step 06 · Pool ALL video folders' fingerprints and self-check (no model). Run once all videos passed cell 7.
# A video with two cows: first set IdentityId per track in its Identities.csv (created on the first run), then rerun.
Stats = BuildPool(OutputRoot, OutputRoot / Folder[6], EmbeddingFolder=Folder[5], ExcludeSeconds=ExcludeSeconds)
display(pd.read_csv(OutputRoot / Folder[6] / "PoolSummary.csv"))
if (OutputRoot / Folder[6] / "SelfCheck.png").exists():
    display(Image(str(OutputRoot / Folder[6] / "SelfCheck.png")))
display(VideoStatus(OutputRoot, Test))

## About this notebook

Each cell calls one function from `CalvingPen/CalvingPipeline/`: `PrepareClips` (VideoTools.py), `DetectCows` (Detect.py), `TrackCows` (Track.py), `CropCows` (Crop.py), `EmbedCrops` (Embed.py), `BuildPool` (Pool.py). What each model does, what we found, and the glossary: `CalvingPen/README.md`.

**Check before moving on:** step 02 `CowShare` per minute (low = YOLO cannot see her); step 03 few long tracks and low `YoloDisagrees`, masks on the cow in the preview; step 04 `Overview.jpg` shows only the cow (no calf, no floor); step 05 bright blocks in the heatmap. Step 06 labels come from each video's `Identities.csv` (edit by hand; never overwritten).